# Mean Average Precision (MAP)
### Specifically, this is a `Ranking Metric`

This this enables us to test the system's ability to retrieve relevant documents from the vector store and return them in order of relevance.
The main advantage of `MAP` is instead of focusing on the number of relevant documents retrieved only, its also rewards the system when relevant
documents are placed in the `first` positions.
In the same spirit, `MAP` also punishes the system when relevant documents are placed in lower ranks/positions.

## Implementation caveats:
- MAP works by calculating the `precision @ k` for relevant documents in a slightly different way:
    Normaly, `precison @ k` is calculated by:
        ```number_of_relevant_documents/k```
    However, in this case:
        ```position_of_relevant_doc/rank * relevance (1 or 0)```
    This ensures that first relevant documents appearing in lower ranks are `punished` i.e., a precision of 1/6 means indicates that the 1st relevant document
    was found in the 6th rank in the retrieved document list while a precision of 1/1 = 1 indicates the 1st relevant document was placed in the 1st rank thus a 
    higher precision of 1.
- Once all `precisions` have been calculated for relevant documents for a qiven question, `AP (Average Precision)` is then the `average` of these `precisions` and 
`MAP` is the `avegare` of `APs` obtained from the set of test questions. 

### Other types of metrics include: 
- Predicive
- Behavioral

In [1]:
import json
from pydantic import BaseModel, Field
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings


TEST_FILE = "tests.jsonl"


class TestQuestion(BaseModel):
    """A test question with expected keywords and reference answer."""

    question: str = Field(description="The question to ask the RAG system")
    keywords: list[str] = Field(description="Keywords that must appear in retrieved context")
    reference_answer: str = Field(description="The reference answer for this question")
    category: str = Field(description="Question category (e.g., direct_fact, spanning, temporal)")


def load_tests() -> list[TestQuestion]:
    """Load test questions from JSONL file."""
    tests = []
    with open(TEST_FILE, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line.strip())
            tests.append(TestQuestion(**data))
    return tests


In [2]:
tests = load_tests()
tests[0]

TestQuestion(question='Who won the prestigious IIOTY award in 2023?', keywords=['Maxine', 'Thompson', 'IIOTY'], reference_answer='Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) award in 2023.', category='direct_fact')

In [3]:
#retriver
# import sentence_transformers

embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
retriever = Chroma(persist_directory="../vector_db/", embedding_function=embeddings).as_retriever()

# testing the retriever
docs = retriever.invoke("Who is Avery?", k=1)
docs[0]


Document(id='b0d7350b-e84c-4d1c-9d64-c6b41108ce57', metadata={'doc_type': 'employees', 'source': 'knowledge-base/employees/Avery Lancaster.md'}, page_content="- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market trends and consumer preferences in the insurance space. This position laid the groundwork for Avery’s future entrepreneurial endeavors.\n\n## Annual Performance History\n- **2015**: **Exceeds Expectations**  \n  Avery’s leadership during Insurellm's foundational year led to successful product launches and securing initial funding.")

In [ ]:
def calculate_average_precision(test: TestQuestion) ->float:
    docs = retriever.invoke(test.question)
    key_precision_scores = []
    doc_ap_scores = []
    for key in test.keywords: 
        key_rel = [1 if key.lower() in doc.page_content.lower() else 0 for doc in docs]
        position = 0
        for rel, rank in zip (key_rel, enumerate(docs, start=1)):
            if rel > 0:
                position += 1
                key_precision_scores += [position/rank[0] * rel]
            else:
                key_precision_scores += [0]
        doc_ap_scores += [sum(key_precision_scores)/sum(key_rel) if sum(key_rel) > 0 else 0.0]
        key_precision_scores.clear()
    return sum(doc_ap_scores)/len(test.keywords) if len(test.keywords) > 0 else 0.0 
    
calculate_average_precision(tests[3])


In [ ]:
def calculate_map() -> float:
    precisions = 0
    for test in tests:
        precisions += calculate_average_precision(test)
    return precisions/len(tests) if len(tests) > 0 else 0.0

In [ ]:
calculate_map()

# Mean Reciprical Rank (MRR) 
### MRR falls under `Ranking Metrics`.
It answers the question `" How good is the system at placing the first relevant item at the top?"`.
MRR is calculated by:
```python
    reciprical_rank = 1 / rank
    mean_reciprical_rank = sum(reciprical_ranks) / len(tests)
```
MRR is usefull in situations where getting a relevant doc or item at the top is essential.
It ignores all other relevant items below the first one making it a good metric where users/queries have few relevant items/docs and we only need to find the first relevant item.

However, due to the above caveat, it's not a sufficient metric where you need to consider the ordering of all relevant items/docs in the entrire list @ K


In [7]:
# implementation 

def calculate_mrr(tests: TestQuestion) ->float:
    test_scores = []
    for test in tests:
        docs = retriever.invoke(test.question)
        rr_scores = []
        for key in test.keywords:
            key_rel = [1 if key.lower() in doc.page_content.lower() else 0 for doc in docs]
            for rank, rel in enumerate(key_rel, start=1):
                if rel == 1:
                    rr_scores += [1/rank]
                    print(f"found {rel} at rank {rank}")
                    break
            print(f"Key: {key}")
            print(f"Key based RR scores: {rr_scores}\n")
        test_scores += [sum(rr_scores) / len(test.keywords)]
        print(f"Test scores for test {tests.index(test)}: {test_scores}")
        print("-"*50)

    return sum(test_scores) / len(tests)



calculate_mrr(tests[3:4])    


found 1 at rank 2
Key: 32
Key based RR scores: [0.5]

found 1 at rank 1
Key: employees
Key based RR scores: [0.5, 1.0]

Test scores for test 0: [0.75]
--------------------------------------------------


0.75

# Normalized Discounted Cummulative Gain (NDCG)
## NDCG fall under `Ranking Metrics`.
NDCG is a ranking metric that works by summing up the discounted `gains` of relevant items/docs in a K-long ranked list.

Discounting is done by multiplying the gain by the reciprical of the log to base 2 of its (rank + 1).
```python
    import math
    discounted_gain = gain / math.log2( rank + 1 )

    cummulative_dg = sum(discounted_gains)
```
Discounted Cummulative Gain `(DCG)` depends on the `length` of the list and the `relevance score type` used, either `binary` or `graded`.
For this reason, DCG cannot be used to compare ranking across lists with varrying lengths and relevance score types. 

Normalized Discounted Cummulative Gain solves this problem by `nomalizing`. Normalizing involves diving the `computed DCG` by `ideal DCG (IDCG)`.

IDCG is obtained by considering a `perfect` ranking for the `same relevance scores`. 
To obtain this:
- Sort the relevance list in decending order and compute DCG for this new list.

Therefore NDCG will be given by:
```python
    ndcg = dcg / idcg
```

The normalized DCG now ranges from 0 to 1 where:
- 1 implies perfect ranking
- 0 impliest least ranking quality (no relevant items in the top K list)
- values approaching 1 imply better ranking quality.

Caveats:
- `Edge case`: If you obtain DCG as 0, then NDCG is also 0, DO NOT compute NDCG since:
```python
    #0/0 is undefined
    0/0 ---> ArithmeticException 
```
- NDCG takes care of `diminishing ranks` by imposing a higher discount on items further down the list.
- Discount imposed between ranks 1 and 2 is higher than between 2 and 3 and so on because of the `log2` function.

NDCG is a proper ranking metric with the following PROS:
- Works with both binary and graded relevance functions
- Considers both Precision and Ranking quality
- It's a normalized metric thus can be used for compaing varrying lists
- Empasises on the importance of `getting the first ranks right` by imposing zero discount on the first rank, and increasing discounts further down the list.


In [8]:
import math

In [9]:
#implementation
#every key is treated an individual test then the average of the keyword metrics represent the test result.
def calculate_ndcg(tests: TestQuestion) ->float:
    scores = []
    for test in tests:
        docs = retriever.invoke(test.question)
        test_ndcg_scores = []
        for key in test.keywords:
            key_rel = [1 if key.lower() in doc.page_content.lower() else 0 for doc in docs]
            discounted_gains = [rel/ math.log2(1 + rank) for rank, rel in enumerate(key_rel, start=1)]
            dcg = sum(discounted_gains)
            if dcg == 0:
                test_ndcg_scores += [0]
            else:
                key_rel.sort(reverse=True)
                key_idg_scores = [rel/ math.log2(1 + rank) for rank, rel in enumerate(key_rel, start=1)]
                test_ndcg_scores += [dcg / sum(key_idg_scores)]
        scores += [sum(test_ndcg_scores) / len(test.keywords)]
    return sum(scores) / len(tests)

calculate_ndcg(tests[3:4])

0.8091944566481509

# Hit Rate @ K
## Hit rate falls under the category of `Ranking Metrics`.
Hit rate answers the question, "What are the odds that a user or query gets atleast 1 relevant item/doc in the top K list ?" 

Hit rate is calculated by taking the total number of tests that had atleast one relevent item divided by the total number of tests.
Hit rate is directly proportional to K since as you go down the list, the odds of a finding atleast one relevant item increases.
For this reason, it's better to monitor hit rate at different K(s) independently.

In [ ]:
def calculate_hit_rate(tests: TestQuestion) -> float:
    scores = []
    for test in tests:
        docs = retriever.invoke(test.question)
        key_scores = []
        for key in test.keywords:
            for doc in docs:
                if key.lower() in doc.page_content.lower():
                    key_scores += [1]
                    break
        scores += [sum(key_scores) / len(test.keywords)]
    return sum(scores) / len(tests)

calculate_hit_rate(tests)
        
